# 11 Shap Explainability

SHAP Explainability for Exoplanet Classifier (4-class)
==========================================================
Suggested notebook name: 11_shap_explainability.ipynb (place in notebooks/)

WHY: A label alone doesn't show why the model decided planet vs EB vs
false_positive vs planet_candidate. SHAP shows which features drove each
individual prediction.

FIX vs the old 2-class version: that script hardcoded "class index 1" as
"the planet class" -- true by coincidence with 2 classes (alphabetically
eclipsing_binary=0, planet=1), but WRONG now that there are 4 classes
(index 1 is false_positive). This version looks up, per star, whichever
class the model actually predicted, and explains that one specifically.

Requires: pip install shap

Paths assume this runs from inside notebooks/.

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

MODEL_PATH = "../models/random_forest.pkl"
LABEL_ENCODER_PATH = "../models/label_encoder.pkl"
FEATURE_COLS_PATH = "../models/feature_cols.pkl"
DATA_PATH = "../outputs/features.csv"
PLOTS_DIR = "../outputs/plots"

os.makedirs(PLOTS_DIR, exist_ok=True)

with open(MODEL_PATH, "rb") as f:
    pipeline = pickle.load(f)
with open(LABEL_ENCODER_PATH, "rb") as f:
    le = pickle.load(f)
with open(FEATURE_COLS_PATH, "rb") as f:
    feature_cols = pickle.load(f)

df = pd.read_csv(DATA_PATH)
df["tic_id"] = df["tic_id"].astype(str).str.replace(".0", "", regex=False)

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing feature columns in {DATA_PATH}: {missing}\nAvailable: {list(df.columns)}")

print(f"Model classes: {list(le.classes_)}")

X = df[feature_cols].fillna(0)
scaler = pipeline.named_steps["scaler"]
clf = pipeline.named_steps["clf"]

# SHAP must see exactly what the classifier sees -- the SCALED features
X_scaled = scaler.transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)

# Which class did the model actually predict for each star? Used below to
# pick the right SHAP slice per star -- not always the same class.
pred_idx_per_row = clf.predict(X_scaled)

explainer = shap.TreeExplainer(clf)
raw_shap = explainer.shap_values(X_scaled)

# Normalize SHAP output into a list: sv_per_class[class_idx] -> (n_samples, n_features)
if isinstance(raw_shap, list):
    sv_per_class = raw_shap
    ev_per_class = explainer.expected_value
elif raw_shap.ndim == 3:
    n_classes = raw_shap.shape[2]
    sv_per_class = [raw_shap[:, :, c] for c in range(n_classes)]
    ev_per_class = explainer.expected_value
else:
    # Binary fallback, shouldn't trigger with 4 classes but kept for safety
    sv_per_class = [-raw_shap, raw_shap]
    ev_per_class = [explainer.expected_value, explainer.expected_value]

if not isinstance(ev_per_class, (list, tuple, np.ndarray)):
    ev_per_class = [ev_per_class] * len(sv_per_class)

# ---- Global summary: multiclass-aware, one color per class ----
plt.figure()
shap.summary_plot(sv_per_class, X_scaled_df, class_names=list(le.classes_), show=False)
plt.tight_layout()
summary_path = f"{PLOTS_DIR}/FIGURE5_shap_summary.png"
plt.savefig(summary_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved global feature importance plot -> {summary_path}")

# ---- Per-star explanation: explain the class the model ACTUALLY predicted ----
saved = 0
for i, row in df.iterrows():
    star_id = row["tic_id"]
    pred_class_idx = int(pred_idx_per_row[i])
    pred_class_name = le.inverse_transform([pred_class_idx])[0]
    try:
        plt.figure()
        shap.force_plot(
            ev_per_class[pred_class_idx],
            sv_per_class[pred_class_idx][i],
            X_scaled_df.iloc[i],
            matplotlib=True,
            show=False,
        )
        plt.tight_layout()
        out_path = f"{PLOTS_DIR}/shap_star_{star_id}_{pred_class_name}.png"
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close()
        saved += 1
    except Exception as e:
        print(f"Skipped star {star_id}: {e}")

print(f"Saved {saved}/{len(df)} per-star SHAP plots to {PLOTS_DIR}/")
print("Each plot explains the SPECIFIC class the model predicted for that star --")
print("filenames include the predicted class so you can tell at a glance.")

d:\isro\Exoplanet_detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model classes: ['eclipsing_binary', 'planet']


d:\isro\Exoplanet_detection\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


Saved global feature importance plot -> ../outputs/plots/FIGURE5_shap_summary.png
Saved 14/14 per-star SHAP plots to ../outputs/plots/
Each plot explains the SPECIFIC class the model predicted for that star --
filenames include the predicted class so you can tell at a glance.


<Figure size 640x480 with 0 Axes>

...

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>